# 🚀 Chapter 13: Deploying scikit-learn Models in Production
**Referensi Buku:** *scikit-learn Cookbook, Third Edition*

---
## 1. Pendahuluan
Setelah model dilatih dan divalidasi, ia harus dikeluarkan dari Jupyter Notebook dan diintegrasikan ke dalam sistem nyata (Production). Bab ini membahas teknik serialisasi dan arsitektur API.

In [ ]:
import numpy as np
import joblib
import os
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Kita latih sebuah pipeline siap pakai
iris = load_iris()
X, y = iris.data, iris.target
prod_pipeline = make_pipeline(StandardScaler(), LogisticRegression())
prod_pipeline.fit(X, y)
print("Model Pipeline berhasil dilatih!")

## 2. Serialization (Menyimpan Model)
Kita bekukan model yang sudah memiliki state (sudah ter-fit) menjadi sebuah file menggunakan `joblib` (lebih disarankan daripada `pickle` untuk model scikit-learn karena efisien untuk numpy array besar).

In [ ]:
model_filename = 'production_model.joblib'

# Simpan ke disk
joblib.dump(prod_pipeline, model_filename)
print(f"Model tersimpan secara lokal di: {model_filename}")

# Di server produksi, kita cukup memuatnya (load)
server_model = joblib.load(model_filename)
print("Model berhasil dimuat kembali di sistem produksi!")

## 3. Mensimulasikan REST API (Konsep)
Di dunia nyata, model yang di-load akan dibungkus oleh *framework* web (seperti Flask atau FastAPI). Di bawah ini adalah simulasi konseptual fungsi API yang menerima data dalam format JSON dan mengembalikan hasil.

In [ ]:
# Simulasi fungsi backend (Flask / FastAPI Route)
def predict_api(json_payload):
    try:
        # 1. Ekstrak data dari request
        input_features = np.array(json_payload['features']).reshape(1, -1)
        
        # 2. Lakukan Prediksi
        prediction = server_model.predict(input_features)[0]
        class_name = iris.target_names[prediction]
        
        # 3. Kembalikan response JSON
        return {"status": "success", "predicted_class": class_name}
    except Exception as e:
        return {"status": "error", "message": str(e)}

# Simulasi Request dari pengguna (misal dari aplikasi mobile)
incoming_request = {
    "features": [5.1, 3.5, 1.4, 0.2]  # Data Setosa baru
}

response = predict_api(incoming_request)
print("Response dari Server API:\n", response)

# Bersihkan file model lokal setelah selesai (opsional)
if os.path.exists(model_filename):
    os.remove(model_filename)

## 4. ONNX (Open Neural Network Exchange)
Sebagai tambahan di buku (opsional), jika model Anda harus di-deploy di sistem yang tidak menggunakan Python (seperti aplikasi Android/Java atau backend C#), Anda disarankan mengonversi model scikit-learn ke format ONNX menggunakan library `skl2onnx`.

---
### 🎉 Selesai!
Ini mengakhiri seluruh seri panduan *scikit-learn Cookbook*. Anda kini memiliki alur kerja lengkap dari Preprocessing, Pemilihan Model, Evaluasi, hingga Deployment.